# Demostración del orquestador multiagente

Esta demo ejecuta el modo local con datos ficticios. No realiza llamadas a OpenAI ni consume créditos.

Ejecutá las celdas en orden desde la carpeta del proyecto.

In [1]:
from decimal import Decimal
from main import build_graph, create_initial_state

# Selección explícita del modo local.
graph = build_graph(mode='local')
print('Grafo local preparado, sin llamadas a OpenAI.')


Grafo local preparado, sin llamadas a OpenAI.


## 1. Solicitud que requiere ambos especialistas

El investigador recupera los registros y el analista calcula la facturación. El supervisor valida antes de finalizar.

In [2]:
query = (
    'Investigá las ventas de llaveros, miniaturas y dummys. '
    'Calculá la facturación de cada producto y el total, '
    'e indicá cuál generó más ingresos.'
)
initial_state = create_initial_state(query)
print(query)


Investigá las ventas de llaveros, miniaturas y dummys. Calculá la facturación de cada producto y el total, e indicá cuál generó más ingresos.


## 2. Flujo de delegación

Cada bloque muestra un nodo ejecutado y su aporte.

In [3]:
final_state = None
for update in graph.stream(
    initial_state,
    config={'recursion_limit': 30},
    stream_mode='updates',
):
    for node, changes in update.items():
        print(f'\nNODO: {node.upper()}')
        if node == 'supervisor':
            print('Destino:', changes['next_agent'])
        print(changes['contributions'][-1]['summary'])
        if node == 'supervisor' and changes['next_agent'] == 'FINISH':
            final_state = changes



NODO: SUPERVISOR
Destino: researcher
Solicitar al investigador datos de ventas con producto, unidades, precio, moneda y fuente.

NODO: RESEARCHER
Investigación realizada sobre ventas ficticias.
- Dummys: 8 unidades, precio unitario ARS 6500.00.
- Llaveros: 50 unidades, precio unitario ARS 1300.00.
- Miniaturas: 12 unidades, precio unitario ARS 4500.00.
Fuente: SQLite, tabla sales.

NODO: SUPERVISOR
Destino: analyst
Solicitar cálculos y conclusiones al analista.

NODO: ANALYST
Análisis de ventas ficticias:
- Dummys: ARS 52000.00 de facturación.
- Llaveros: ARS 65000.00 de facturación.
- Miniaturas: ARS 54000.00 de facturación.
Unidades totales: 70.
Facturación total: ARS 171000.00.
Mayor facturación: Llaveros.
La facturación corresponde a unidades × precio unitario. No representa ganancia, porque no contamos con costos.

NODO: SUPERVISOR
Destino: FINISH
Datos y cálculos validados. Finalizar la tarea.

Resultado final — datos ficticios de demostración

Investigación realizada sobre vent

## 3. Respuesta final y registro de aportes

In [4]:
assert final_state is not None
print(final_state['final_answer'])
print('\nCompletada:', final_state['task_completed'])
print('Intervenciones:', final_state['steps'])

agents = [item['agent'] for item in final_state['contributions']]
print('Orden de participación:', agents)
assert final_state['task_completed'] is True
assert agents == [
    'supervisor', 'researcher', 'supervisor', 'analyst', 'supervisor'
]


Resultado final — datos ficticios de demostración

Investigación realizada sobre ventas ficticias.
- Dummys: 8 unidades, precio unitario ARS 6500.00.
- Llaveros: 50 unidades, precio unitario ARS 1300.00.
- Miniaturas: 12 unidades, precio unitario ARS 4500.00.
Fuente: SQLite, tabla sales.

Análisis de ventas ficticias:
- Dummys: ARS 52000.00 de facturación.
- Llaveros: ARS 65000.00 de facturación.
- Miniaturas: ARS 54000.00 de facturación.
Unidades totales: 70.
Facturación total: ARS 171000.00.
Mayor facturación: Llaveros.
La facturación corresponde a unidades × precio unitario. No representa ganancia, porque no contamos con costos.

Validación del supervisor: los datos tienen los campos requeridos y los resultados coinciden con el recálculo de unidades × precio unitario.

Completada: True
Intervenciones: 5
Orden de participación: ['supervisor', 'researcher', 'supervisor', 'analyst', 'supervisor']


## 4. Comprobación de los resultados

Ejecutamos otra consulta para comprobar los importes del estado completo.

In [5]:
result = graph.invoke(
    create_initial_state(query),
    config={'recursion_limit': 30},
)
analysis = result['analysis_results']
assert Decimal(analysis['total_revenue']) == Decimal('171000')
assert analysis['total_units'] == 70
assert analysis['top_products'] == ['Llaveros']
print('Resultados esperados verificados.')


Resultados esperados verificados.


## 5. Consulta sin datos disponibles

Una consulta fuera del catálogo debe terminar sin afirmar que la tarea fue completada.

In [6]:
missing = graph.invoke(
    create_initial_state('Investiga bicicletas'),
    config={'recursion_limit': 30},
)
print(missing['final_answer'])
print('Completada:', missing['task_completed'])
assert missing['task_completed'] is False


No encontré datos para responder la consulta. Esta demostración contiene ventas ficticias de llaveros, miniaturas y dummys.
Completada: False


## Alcance de la demostración

Se demuestra la delegación, el uso de herramientas, el registro de aportes y la finalización del modo local. El modo con API está implementado en el proyecto, pero no se ejecuta en este notebook.